# Generate one rigid population for E1 public continuity

Prospective qualification for `uibcdf/dockingmt#78`. Run only after registering
this notebook, installed runtime, immutable inputs and the external resource
supervisor. Set `DOCKINGMT_CONTINUITY_ROOT` and a fresh
`DOCKINGMT_CONTINUITY_OUTPUT`.

This is a new declared experiment. It reuses the received full18 pose-1 chemistry
and indexed H, assigns Gasteiger–Marsili charges and AutoDock4
`chemical_environment` types once, then prepares an explicitly rigid ligand.
A second preparation tests public reconstruction and scientific idempotence;
report histories may legitimately differ. Inspection sidecars are evidence,
never a molecular-mechanics restoration format.

The single frozen search uses Vina 1.2.7, CPU 1, seed 7, exhaustiveness 1,
maximum 3 poses and an energy range of 3 kcal/mol in the original 20 Å box.
The actual returned population is saved before validation. Fewer than two poses
means **INCOMPLETE**, with no retry, alternate seed or parameter adjustment.
No fixed scoring, comparison, H generation or separate optimization is requested here.
Docking retains the optimization inherent to its native search.
Later rescoring/comparison is a separate registered phase; its scoring box may
be explicitly larger. This notebook does not close E1 or the earlier incomplete
flexible two-pose experiment.


In [ ]:
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
import sys
from copy import deepcopy
from datetime import datetime, timezone
from functools import wraps
from pathlib import Path, PurePosixPath

root = Path(os.environ['DOCKINGMT_CONTINUITY_ROOT']).resolve()
output = Path(os.environ['DOCKINGMT_CONTINUITY_OUTPUT']).resolve()
assert not output.exists() and not root.is_relative_to(output)
assert os.sched_getaffinity(0) == {0}
assert os.getpriority(os.PRIO_PROCESS, 0) >= 10
output.mkdir(parents=True, exist_ok=False)
evidence = root / 'devguide/validation/data/e1_complete_pose_2026-11-06'
trusted_root = '45b3665ba09b6979aa1bcdf1bea0ac3e8102f081fefbfc51348c55bc71405df3'
source_state = 'e1-rigid-continuity-benzamidinium-20261107'
receptor_state = 'e1-captured-3PTB-receptor-20261020'


def file_sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def available_ram():
    fields = dict(
        line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines()
    )
    return int(fields['MemAvailable'].split()[0]) * 1024


assert available_ram() >= 8 * 1024**3
manifest_path = evidence / 'manifest.json'
assert file_sha(manifest_path) == trusted_root
manifest = json.loads(manifest_path.read_text())
assert manifest['owner'] == 'uibcdf/dockingmt#78'
stored_inputs = {str(manifest_path): trusted_root}
for row in manifest['files']:
    relative = PurePosixPath(row['path'])
    assert not relative.is_absolute() and '..' not in relative.parts
    assert str(relative) == row['path']
    path = evidence / relative
    assert path.resolve().is_relative_to(evidence.resolve())
    assert path.stat().st_size == row['bytes'] and file_sha(path) == row['sha256']
    stored_inputs[str(path)] = row['sha256']
    if row['path'].endswith('.gz'):
        digest = hashlib.sha256()
        size = 0
        with gzip.open(path, 'rb') as stream:
            while chunk := stream.read(1024 * 1024):
                digest.update(chunk)
                size += len(chunk)
        assert size == row['decoded_bytes']
        assert digest.hexdigest() == row['decoded_sha256']


def input_bytes(name):
    rows = [
        r
        for r in manifest['files']
        if r['path'] == name or r.get('original_path') == name
    ]
    assert len(rows) == 1, name
    row = rows[0]
    data = (evidence / row['path']).read_bytes()
    assert hashlib.sha256(data).hexdigest() == row['sha256']
    return gzip.decompress(data) if row['path'].endswith('.gz') else data


inputs = output / 'inputs'
inputs.mkdir()
source_path = inputs / 'full18_chemical_only.h5msm'
source_path.write_bytes(input_bytes('normal/pose_1/full18_chemical_only.h5msm'))
assert (
    file_sha(source_path)
    == 'efdaab5cdc185ef9f5b68ca018a092729455ca833172ca48b5afd40f95b024f3'
)
source_transport = evidence / 'normal/pose_1/full18_chemical_only.h5msm.gz'
assert (
    file_sha(source_transport)
    == 'bc81efd6a6d6de00141fb03473970a28f356c54bf95eefdf769fd972229bc468'
)
receptor_path = inputs / 'provided_receptor.pdbqt'
receptor_path.write_bytes(input_bytes('provided_receptor.pdbqt'))
assert (
    file_sha(receptor_path)
    == 'edeff77e6222c5c5ab1aa6be2354efc323c0ffe8a7884366069ee0addbf7b970'
)
contract = json.loads(input_bytes('prospective_scoring_contract.json'))
assert contract['receptor_sha256'] == file_sha(receptor_path)
center = contract['original_backend_box']['center']
assert center == [-1.759111, 14.461, 16.915778]
assert contract['original_backend_box']['size'] == [20.0, 20.0, 20.0]
assert contract['original_backend_box']['unit'] == 'angstrom'
local_inputs_before = {str(p): file_sha(p) for p in (source_path, receptor_path)}
assert metadata.version('vina') == '1.2.7'

In [ ]:
import molsysmt as msm
import numpy as np
import pyunitwizard as puw

import dockingmt as dmt
from dmt_test_support.qualify_chemical_templates import snapshot

budgets = dict(
    partial_charge_assignment=1,
    atom_type_assignment=1,
    prepare_ligand=2,
    to_molecular_system=1,
    docking_attempts=1,
    docking_returns=1,
    graph_witnesses=2,
    charge_audits=2,
    writer_exports=3,
    forbidden_operations=0,
)
counts = dict.fromkeys(budgets, 0)
patches = []
writer_exports = []
phase = 'preparation'
caller_policy = deepcopy(puw.configure.report())


def plain(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if puw.is_quantity(value):
        return puw.convert(value, to_form='record').to_dict()
    raise TypeError(type(value).__name__)


def normalized(value):
    return json.loads(json.dumps(value, default=plain, allow_nan=False))


def save(path, value):
    with path.open('x') as stream:
        json.dump(value, stream, default=plain, indent=2, allow_nan=False)
        stream.write('\n')


def event(operation, **fields):
    with (output / 'operations.jsonl').open('a') as stream:
        stream.write(
            json.dumps(
                dict(
                    operation=operation,
                    phase=phase,
                    at_utc=datetime.now(timezone.utc).isoformat(),
                    counts=counts.copy(),
                    **fields,
                ),
                allow_nan=False,
            )
            + '\n'
        )


def consume(name):
    assert counts[name] < budgets[name], name
    counts[name] += 1
    event(name)


def counted(function, name):
    @wraps(function)
    def wrapped(*args, **kwargs):
        consume(name)
        return function(*args, **kwargs)

    return wrapped


def forbidden(*args, **kwargs):
    counts['forbidden_operations'] += 1
    event('forbidden_operation')
    raise AssertionError(
        'H generation, fixed scoring and comparison are forbidden in generation'
    )


def replace(owner, name, value):
    patches.append((owner, name, getattr(owner, name)))
    setattr(owner, name, value)


original_convert = msm.convert


@wraps(original_convert)
def monitored_convert(*args, **kwargs):
    is_writer = kwargs.get('to_form') == 'string:pdbqt_text' and kwargs.get(
        'return_atom_map'
    )
    if is_writer:
        consume('writer_exports')
    result = original_convert(*args, **kwargs)
    if is_writer:
        text, atom_map = result
        payload = text.removeprefix('pdbqt_text:').encode()
        record = normalized(atom_map)
        writer_exports.append(dict(phase=phase, atom_map=record))
        save(output / f'writer_export_{len(writer_exports)}.json', record)
        assert record['schema'] == 'molsysmt.atom_export@1'
        assert record['payload_sha256'] == hashlib.sha256(payload).hexdigest()
        assert record['n_source_atoms'] == record['n_output_atoms'] == 13
        assert record['output_to_source_atom_indices'] == list(range(13))
        assert record['source_to_output_atom_indices'] == list(range(13))
    return result


replace(msm, 'convert', monitored_convert)
replace(
    msm.build,
    'assign_partial_charges',
    counted(msm.build.assign_partial_charges, 'partial_charge_assignment'),
)
replace(
    msm.build,
    'assign_autodock_atom_types',
    counted(msm.build.assign_autodock_atom_types, 'atom_type_assignment'),
)
replace(dmt, 'prepare_ligand', counted(dmt.prepare_ligand, 'prepare_ligand'))
replace(
    dmt.PreparedLigand,
    'to_molecular_system',
    counted(dmt.PreparedLigand.to_molecular_system, 'to_molecular_system'),
)
replace(dmt, 'dock', counted(dmt.dock, 'docking_attempts'))
replace(
    msm.chemical_states,
    'get_atom_correspondences',
    counted(msm.chemical_states.get_atom_correspondences, 'graph_witnesses'),
)
replace(
    dmt,
    'audit_preparation_charges',
    counted(dmt.audit_preparation_charges, 'charge_audits'),
)
replace(msm.build, 'add_missing_hydrogens', forbidden)
replace(dmt, 'score', forbidden)
replace(dmt, 'compare_scores', forbidden)
replace(dmt.VinaBackend, 'score', forbidden)


def inspect_native(system):
    # Snapshot is the existing staged qualification helper; these are inspections,
    # not an MM codec or a recipe for restoring reports.
    charges = (
        msm.get(system, element='atom', partial_charge=True)
        if msm.has_attribute(system, 'partial_charge')
        else None
    )
    if puw.is_quantity(charges):
        charges = puw.get_value(charges, to_unit='elementary_charge')
    return normalized(
        dict(
            ids=[str(v) for v in msm.get(system, element='atom', atom_id=True)],
            names=[str(v) for v in msm.get(system, element='atom', name=True)],
            elements=list(
                msm.get(
                    system,
                    element='atom',
                    atom_type=True,
                    chemical_state='structure',
                    structure_indices=0,
                )
            ),
            types=list(msm.get(system, element='atom', atom_ff_type=True))
            if msm.has_attribute(system, 'atom_ff_type')
            else None,
            charges_e=charges,
            coordinates_nm=puw.get_value(
                msm.get(system, coordinates=True), to_unit='nm'
            ),
            chemistry_structures=snapshot(system),
            mechanics=system.molecular_mechanics.to_dict(),
            partial_charge_assignment=system.molecular_mechanics.partial_charge_assignment,
            atom_type_assignment=system.molecular_mechanics.atom_type_assignment,
            native_charge_unit='elementary_charge',
        )
    )


def inspect_prepared(prepared):
    return normalized(
        dict(
            declaration=prepared.to_dict(),
            coordinates_nm=puw.get_value(prepared.coordinates, to_unit='nm'),
            native=inspect_native(prepared.source_molsys),
            scope='Inspection only; never a restoration input',
        )
    )


runtime = {}
for name, module in (
    ('dockingmt', dmt),
    ('molsysmt', msm),
    ('pyunitwizard', puw),
    ('numpy', np),
):
    path = Path(module.__file__).resolve()
    runtime[name] = dict(
        version=metadata.version(name), path=str(path), sha256=file_sha(path)
    )
helper_path = Path(snapshot.__code__.co_filename).resolve()
runtime['snapshot_helper'] = dict(path=str(helper_path), sha256=file_sha(helper_path))
assert file_sha(helper_path) == file_sha(
    root / 'dmt_test_support/qualify_chemical_templates.py'
)
full18 = msm.convert(str(source_path), to_form='molsysmt.MolSys')
assert msm.get(full18, n_atoms=True) == 18
full18_before = inspect_native(full18)
assert full18_before['elements'].count('H') == 9
save(output / 'source18_inspection.json', full18_before)
charged = msm.build.assign_partial_charges(full18, method='gasteiger_marsili')
typed = msm.build.assign_autodock_atom_types(
    charged, typing_scheme='autodock4', method='chemical_environment'
)
typed_before = inspect_native(typed)
save(output / 'assigned18_inspection.json', typed_before)
assert inspect_native(full18) == full18_before
source_prepared = dmt.prepare_ligand(
    typed, selection='all', state_id=source_state, active_torsion_bonds=[]
)
source_before = inspect_prepared(source_prepared)
save(output / 'source_prepared_inspection.json', source_before)
source_pdbqt = source_prepared.to_pdbqt()
(output / 'source_prepared.pdbqt').write_text(source_pdbqt)
reconstructed = source_prepared.to_molecular_system()
reconstructed_before = inspect_native(reconstructed)
save(output / 'reconstructed13_inspection.json', reconstructed_before)
gate_prepared = dmt.prepare_ligand(
    reconstructed, selection='all', state_id=source_state, active_torsion_bonds=[]
)
gate_before = inspect_prepared(gate_prepared)
save(output / 'gate_prepared_inspection.json', gate_before)
gate_pdbqt = gate_prepared.to_pdbqt()
(output / 'gate_prepared.pdbqt').write_text(gate_pdbqt)

# Scientific idempotence compares chemistry and numerical fields. Provider report
# histories and source/retention histories may legitimately differ.
for field in (
    'state_id',
    'n_atoms',
    'atom_names',
    'group_names',
    'group_ids',
    'atom_types',
    'torsion_dof',
    'pdbqt_atom_indices',
):
    assert source_before['declaration'][field] == gate_before['declaration'][field], (
        field
    )
np.testing.assert_allclose(
    source_before['coordinates_nm'], gate_before['coordinates_nm'], rtol=0, atol=1e-12
)
np.testing.assert_allclose(
    source_prepared.charges, gate_prepared.charges, rtol=0, atol=1e-12
)
for field in ('ids', 'names', 'elements', 'types'):
    assert source_before['native'][field] == gate_before['native'][field], field
for field in ('topology', 'chemical_states', 'structure_chemical_state_indices'):
    assert (
        source_before['native']['chemistry_structures'][field]
        == gate_before['native']['chemistry_structures'][field]
    ), field

witnesses = []
for label, prepared, text in (
    ('source', source_prepared, source_pdbqt),
    ('gate', gate_prepared, gate_pdbqt),
):
    assert prepared.n_atoms == 13 and prepared.torsion_dof == 0
    assert source_before['native']['elements'].count('H') == 4
    assert prepared.pdbqt_atom_indices == list(range(13))
    assert prepared.metadata['active_torsion_bonds'] == []
    assert prepared.metadata['torsion_policy'] == 'rigid_only'
    tree = normalized(msm.form.string_pdbqt_text.get_torsion_tree('pdbqt_text:' + text))
    save(output / f'{label}_torsion_tree.json', tree)
    assert tree['torsdof'] == 0 and tree['fragment_offsets'] == [0, 13]
    assert tree['fragment_atom_indices'] == list(range(13))
    assert tree['branch_atom_pairs'] == tree['branch_fragment_pairs'] == []
    audit = dmt.audit_preparation_charges(prepared)
    save(output / f'{label}_charge_audit.json', audit)
    assert audit['assessment'] == 'consistent'
    native = prepared.source_molsys
    witness = msm.chemical_states.get_atom_correspondences(
        native,
        reference_molecular_system=native,
        structure_indices=0,
        reference_structure_indices=0,
        chemical_state='structure',
        reference_chemical_state='structure',
        candidate_atom_correspondences=[[[i, i] for i in range(13)]],
        max_matches=1,
    )
    witness = normalized(witness)
    save(output / f'{label}_self_witness.json', witness)
    assert witness['search_complete'] is True and witness['n_candidate_matches'] == 1
    np.testing.assert_array_equal(
        witness['atom_correspondences'], [[[i, i] for i in range(13)]]
    )
    assert witness['source_graph_digest'] == witness['reference_graph_digest']
    witnesses.append(witness)
assert witnesses[0]['source_graph_digest'] == witnesses[1]['source_graph_digest']
assert inspect_native(full18) == full18_before
assert inspect_native(typed) == typed_before
assert inspect_native(reconstructed) == reconstructed_before
assert inspect_prepared(source_prepared) == source_before
assert inspect_prepared(gate_prepared) == gate_before
assert puw.configure.report() == caller_policy
save(
    output / 'idempotence_gate.json',
    dict(
        status='PASS',
        scope='Scientific fields; no literal equality of report histories',
        source_graph_digest=witnesses[0]['source_graph_digest'],
        gate_graph_digest=witnesses[1]['source_graph_digest'],
        coordinate_tolerance_nm=1e-12,
        charge_tolerance_e=1e-12,
        source_pdbqt_sha256=hashlib.sha256(source_pdbqt.encode()).hexdigest(),
        gate_pdbqt_sha256=hashlib.sha256(gate_pdbqt.encode()).hexdigest(),
        counts=counts.copy(),
    ),
)

In [ ]:
phase = 'single_frozen_docking'
assert available_ram() >= 8 * 1024**3
assert os.sched_getaffinity(0) == {0} and os.getpriority(os.PRIO_PROCESS, 0) >= 10
source_domain = dmt.BoxRegion(
    puw.quantity(center, 'angstrom'), puw.quantity([20.0, 20.0, 20.0], 'angstrom')
)
problem = dmt.DockingProblem(
    receptor_path,
    source_prepared,
    source_domain,
    metadata=dict(
        partner_state_id=source_state,
        receptor_state_id=receptor_state,
        experiment='e1_rigid_continuity_generation',
        preparation_source='Received full18 pose1, new named Q/types, no new H',
    ),
)
protocol = dmt.VinaProtocol(
    cpu=1,
    seed=7,
    exhaustiveness=1,
    n_poses=3,
    energy_range=puw.quantity(3.0, 'kcal/mol'),
    scoring='vina',
    capture_backend_inputs=True,
    collect_timings=True,
)
save(output / 'source_problem.json', problem.to_dict())
save(output / 'source_protocol.json', protocol.to_dict())
source_result = dmt.dock(problem, protocol)
source_record = source_result.to_dict()
# Preserve the complete actual return before all qualification assertions.
save(output / 'source_result.json', source_record)
consume('docking_returns')
actual_n = len(source_result)
status = 'PASS' if 2 <= actual_n <= 3 else 'INCOMPLETE' if actual_n < 2 else 'FAILED'
save(
    output / 'population.json',
    dict(
        status=status,
        requested_maximum=3,
        actual_returned=actual_n,
        retry_permitted=False,
    ),
)
assert actual_n <= 3
assert (
    normalized(json.loads(json.dumps(source_record, allow_nan=False))) == source_record
)
restored = dmt.DockingResult.from_dict(deepcopy(source_record))
assert restored.to_dict() == source_record
collection_audit = dmt.audit_result(source_record)
save(output / 'source_audit.json', collection_audit)
if actual_n < 2:
    # A zero-pose return has no pose-reading report and its collection audit may
    # be incomplete. Preserve that actual outcome without manufacturing evidence.
    assert inspect_native(full18) == full18_before
    assert inspect_native(typed) == typed_before
    assert inspect_native(reconstructed) == reconstructed_before
    assert inspect_prepared(source_prepared) == source_before
    assert inspect_prepared(gate_prepared) == gate_before
    assert puw.configure.report() == caller_policy
    for path, digest in {**stored_inputs, **local_inputs_before}.items():
        assert file_sha(Path(path)) == digest
    assert counts == budgets
    save(
        output / 'receipt.json',
        dict(
            schema='dockingmt.e1_rigid_continuity_generation@1',
            owner='uibcdf/dockingmt#78',
            status='INCOMPLETE',
            trusted_construction_manifest_sha256=trusted_root,
            immutable_inputs=stored_inputs,
            local_input_sha256=local_inputs_before,
            runtime=runtime,
            caller_unit_policy=caller_policy,
            budgets=budgets,
            counts=counts,
            requested_maximum=3,
            actual_returned=actual_n,
            retries=0,
            retry_permitted=False,
            partner_state_id=source_state,
            receptor_state_id=receptor_state,
            actual_collection_audit_status=collection_audit['status'],
            new_named_charge_assignments=1,
            new_named_type_assignments=1,
            new_preparations=2,
            searches=1,
            fixed_score_calls=0,
            comparisons=0,
            new_hydrogen_calls=0,
            source_integrity_preserved=True,
            continuation_requires_separate_registered_receiving=True,
            full_e1_accepted=False,
        ),
    )
    for owner, name, original in reversed(patches):
        setattr(owner, name, original)
    raise RuntimeError(
        'INCOMPLETE: fewer than two poses returned; retain this attempt and stop without retry'
    )
assert collection_audit['status'] == 'consistent'
artifacts = source_record['provenance']['backend_artifacts']
save(output / 'captured_input_verification.json', dmt.verify_captured_inputs(artifacts))
assert (
    artifacts['partner']['sha256'] == hashlib.sha256(source_pdbqt.encode()).hexdigest()
)
assert artifacts['receptor']['sha256'] == file_sha(receptor_path)
assert (
    writer_exports[-1]['atom_map']['payload_sha256'] == artifacts['partner']['sha256']
)
reading = source_record['provenance']['backend_output']['pose_reading']
assert reading['method'] == 'molsysmt.form.string_pdbqt_text.read_poses'
assert reading['report']['schema'] == 'molsysmt.pdbqt_pose_read@1'
assert source_record['provenance']['backend_box'] == dict(
    center=center,
    size=[20.0, 20.0, 20.0],
    unit='angstrom',
    rounding_decimals=6,
)
assert source_record['provenance']['backend_version'] == '1.2.7'
for index, pose in enumerate(source_result, 1):
    assert pose.pose_id == f'pose_{index}' and pose.rank == index
    assert (
        pose.partner_state_id == source_state
        and pose.receptor_state_id == receptor_state
    )
    assert (
        set(pose.scores)
        == set(pose.score_definitions)
        == {'vina', 'inter', 'intra', 'torsion'}
    )
    assert np.isfinite(list(pose.scores.values())).all()
    assert pose.metadata['pose_atom_order'] == 'verified_pdbqt_order'
    assert pose.metadata['prepared_atom_indices'] == list(range(13))
    assert pose.metadata['selected_atom_indices'] == list(range(13))
    assert pose.metadata['selected_partner_n_atoms'] == 13
    keys = pose.metadata['source_atom_keys']
    assert len(keys) == 13
    assert [k['atom_id'] for k in keys] == source_before['native']['ids']
    assert [k['atom_name'] for k in keys] == source_before['native']['names']
    assert [k['element'] for k in keys] == source_before['native']['elements']
    assert np.isfinite(puw.get_value(pose.coordinates, to_unit='nm')).all()
    save(output / f'pose_{index}_original.json', pose.to_dict())
history = source_result.ranking_history
assert len(history) == 1
assert history[0]['origin'] == 'backend' and history[0]['evidence'] == 'recorded'
assert history[0]['score_name'] == 'vina' and history[0]['ascending'] is True
assert history[0]['output_order'] == list(range(actual_n))
assert [row['pose_id'] for row in history[0]['input_order']] == [
    p.pose_id for p in source_result
]
assert [row['value'] for row in history[0]['input_order']] == [
    p.scores['vina'] for p in source_result
]
save(output / 'original_ranking_history.json', history)
assert inspect_native(full18) == full18_before
assert inspect_native(typed) == typed_before
assert inspect_native(reconstructed) == reconstructed_before
assert inspect_prepared(source_prepared) == source_before
assert inspect_prepared(gate_prepared) == gate_before
assert source_result.to_dict() == source_record
assert puw.configure.report() == caller_policy
for path, digest in {**stored_inputs, **local_inputs_before}.items():
    assert file_sha(Path(path)) == digest
assert counts == budgets
assert len(writer_exports) == 3
for name in ('rdkit', 'rdkit.Chem.rdchem', 'rdkit.Chem.rdPartialCharges'):
    module = sys.modules[name]
    path = Path(module.__file__).resolve()
    runtime[name] = dict(path=str(path), sha256=file_sha(path))
save(
    output / 'receipt.json',
    dict(
        schema='dockingmt.e1_rigid_continuity_generation@1',
        owner='uibcdf/dockingmt#78',
        status=status,
        trusted_construction_manifest_sha256=trusted_root,
        immutable_inputs=stored_inputs,
        local_input_sha256=local_inputs_before,
        runtime=runtime,
        caller_unit_policy=caller_policy,
        budgets=budgets,
        counts=counts,
        requested_maximum=3,
        actual_returned=actual_n,
        retries=0,
        partner_state_id=source_state,
        receptor_state_id=receptor_state,
        full18_hydrogens='Received indexed inventory; no generation',
        new_named_charge_assignments=1,
        new_named_type_assignments=1,
        new_preparations=2,
        searches=1,
        fixed_score_calls=0,
        comparisons=0,
        new_hydrogen_calls=0,
        original_scores_per_pose=4,
        original_ranks_and_backend_history_preserved=True,
        prepared_sidecars='Inspection only; reproduce assignments via public APIs, never restore MM reports',
        continuation_requires_separate_registered_receiving=True,
        full_e1_accepted=False,
    ),
)
for owner, name, original in reversed(patches):
    setattr(owner, name, original)
assert 2 <= actual_n <= 3